In [0]:
# Databricks notebook source
# Scheduled entry point for the upload pipeline.
#
# Runs every 15 minutes. Discovers new landing files, validates them
# against their dataset contract, and loads the valid ones into bronze.
#
# All the logic lives in ea_pipeline.orchestrate — this file exists only
# to give the job something to point at.


from ea_pipeline.orchestrate import release_stale_claims, run_upload_pipeline

# COMMAND ----------

# Release rows abandoned by an interrupted run BEFORE processing, so
# this run picks them up rather than leaving them until next time.
release_stale_claims()

summary = run_upload_pipeline()

print(summary)

if summary["errors"] > 0:
    raise RuntimeError(f"Pipeline run had {summary['errors']} errors: {summary}")

# COMMAND ----------

# Surface the result to the job run history, so a run's outcome is
# visible without opening the notebook output.
dbutils.notebook.exit(str(summary))